In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import pandas as pd
import random

model_name = "mdhugol/indonesia-bert-sentiment-classification"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:

nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
print("Model berhasil dimuat! Lanjut proses labeling...")

df = pd.read_csv('../data/jnt_reviews_final.csv')

def auto_label(row):
    score = row['score']
    text = str(row['content'])[:512]
    
    if score >= 4:
        return 0 
    elif score <= 2:
        return 1 
    else: 
        res = nlp(text)[0]
        return 1 if res['label'] == 'LABEL_2' else 0

df['label'] = df.apply(auto_label, axis=1)

In [ ]:
#  Balancing
df_aman = df[df['label'] == 0]
df_churn = df[df['label'] == 1]

print(f"\nInitial Label Count:")
print(f"Aman : {len(df_aman)}")
print(f"Churn: {len(df_churn)}")

max_churn = len(df_aman) * 2
df_churn_sampled = df_churn.sample(n=min(len(df_churn), max_churn), random_state=42)

df_final = pd.concat([df_aman, df_churn_sampled]).sample(frac=1, random_state=42)
print(df_final['label'].value_counts())

df_final.to_csv('../data/processed/jnt_final_labeled.csv', index=False)